# Section 2 — Foundations of AI Pipelines

## Notebook 1 · The feature pipeline

*Estimated reading + run time: 45 minutes. Several runnable code cells. The reader is expected to step through these.*

By the end of this notebook you will have:

- A clean, prepared train / validation / test split of the Chicco & Jurman heart-failure dataset, ready for the training-pipeline notebook.
- A working knowledge of *what* a feature pipeline does, *what is modifiable* at each stage, and *where the pitfalls are* — using the live event's most cited teaching moment (the `time` column data leakage) as the worked example.
- A saved `prepared_heart_failure.joblib` artefact that the training-pipeline notebook will load. (If you skip this notebook and go straight to notebook 02, it rebuilds the artefact from scratch — but you will miss the lessons that make the rebuild worth doing once.)

**Why three notebooks instead of one.** A real clinical AI deployment splits along the same three seams these notebooks do: a *feature pipeline* that owns getting clinical data from its source into a model-ready shape; a *training pipeline* that owns model selection and evaluation; an *inference pipeline* that owns serving the model and producing clinical output. The boundary between these is the most important architectural decision in clinical AI. Section 2 makes the boundary explicit so Section 3's challenges can re-point the same architecture at different clinical questions.

## 1. What this stage owns

The feature pipeline takes raw clinical data and produces *features* — the numeric inputs a model can learn from. Specifically, it owns five jobs:

| Job | What it does | Where it can fail |
| --- | --- | --- |
| **Source the data** | Load from a research dataset, an EPR export, or a real-time feed. | Wrong dataset version, broken authentication, untracked schema change. |
| **Inspect for missingness, ranges, and dtypes** | Confirm the data is shaped as expected before anything else happens. | Silent NaN, locale-flipped decimals, mixed-unit columns. |
| **Identify and remove leakage** | Drop columns the model could not legitimately have at prediction time. | The famous one — the `time` column in this dataset. |
| **Handle class imbalance** | Decide how the minority class will be represented in training. | Default settings produce an accuracy-but-no-sensitivity model. |
| **Split** | Train / validation / test, stratified by outcome so each split has the same prevalence. | Random splits across patients with multiple records can leak across folds. |

**Anything before the model is trained lives here.** That includes scaling, encoding, imputation, and feature engineering. The output of this notebook is the input to notebook 02. The interface between the two is a single saved file.

## 2. Source the data

The same load pattern as Section 0 and Chapter 06 — HuggingFace primary, UCI fallback. Pinning the dataset and column names per master-brief §7.

In [ ]:
import pandas as pd
import numpy as np

try:
    from datasets import load_dataset
    df = load_dataset("mstz/heart_failure", split="train").to_pandas()
    source = "HuggingFace"
except Exception as e:
    print(f"  HuggingFace unavailable ({type(e).__name__}). Falling back to UCI...")
    df = pd.read_csv(
        "https://archive.ics.uci.edu/ml/machine-learning-databases/00519/"
        "heart_failure_clinical_records_dataset.csv"
    )
    source = "UCI fallback"

# Pin canonical column names (master brief §7).
TARGET   = "DEATH_EVENT" if "DEATH_EVENT" in df.columns else "is_dead"
TIME_COL = "time"        if "time"        in df.columns else "TIME"

print(f"Loaded {len(df)} rows, {df.shape[1]} columns from: {source}")
print(f"Target column:    {TARGET}")
print(f"Follow-up column: {TIME_COL}  (will be dropped as a leakage source — see §4)")
df.head(3)

## 3. Inspect — missingness, ranges, dtypes

Three things to confirm before anything else happens. *Inspecting* a clinical dataset before modelling is the single highest-leverage habit in clinical ML.

★ **Pause-and-predict.** What kinds of issues do you expect to find? Is this dataset clean (publicly curated) or messy (real EPR export)?

In [ ]:
# Three quick inspections.
print("=== Dtypes ===")
print(df.dtypes.to_string())

print("\n=== Missingness (per column) ===")
missing = df.isna().sum()
if (missing == 0).all():
    print("  No missing values in any column. (Curated public dataset — real EPR")
    print("  exports are rarely this clean. The handling pattern shown below applies)")
    print("  even when missingness is zero today, because the same pipeline runs)")
    print("  tomorrow on data that may not be.)")
else:
    print(missing[missing > 0].to_string())

print("\n=== Ranges (numeric columns only) ===")
print(df.describe().T[["min", "max", "mean", "std"]].round(2).to_string())

Three things to register from that output. The dataset is fully numeric — this is unusual for clinical data and is one reason why this dataset is so frequently used as a teaching example. Missingness is zero — also unusual, also pedagogically convenient. The ranges across columns vary by orders of magnitude — `platelets` is in the hundreds of thousands, `serum_sodium` is in the 130s, `anaemia` is 0/1. **Different scales matter for some models (logistic regression, SVM) and not for others (random forest, gradient boosting).** Section 5 of this notebook handles scaling explicitly.

**A note on real-world clinical data.** EPR exports almost always have missingness — a vital sign not recorded, a lab result pending, a coded field left blank. Production feature pipelines handle missingness with one of three strategies: (1) drop rows with missing critical fields, (2) impute (mean / median / model-based), (3) treat missingness as a signal in itself (a column that is missing for a reason). The choice is clinical, not technical — *why* is this missing? *Was the test not done because the patient was too well, or because the patient was too unwell?* Different answers, different imputation strategies.

## 4. Identify and remove leakage — the `time` column

Chapter 06 introduced this; here we resolve it as code. The `time` column is the *follow-up duration* — how long the patient was observed after enrolment. It is *not* a feature available at the time of prediction. It is recorded *retrospectively* once the patient has either died or been censored.

**Train a model that uses `time` as a feature, and it learns a near-perfect rule:** *long follow-up time and still alive → low mortality risk; short follow-up time and dead → high mortality risk.* The rule is mathematically true and clinically useless, because at admission the model does not yet know how long the patient will be observed. The cell below removes the column, with the explanation embedded in the code so future readers (including future you) cannot miss it.

In [ ]:
# Drop the follow-up time column. It is a label leak: `time` records what happened
# *after* the prediction window, so a model that uses it cheats. See master brief §7
# ("Heart-failure 'time column' leakage as the worked example").
leaky_cols = [TIME_COL]
df_clean = df.drop(columns=leaky_cols)

# Separate features from target.
y = df_clean[TARGET]
X = df_clean.drop(columns=[TARGET]).select_dtypes(include="number")

print(f"Removed {len(leaky_cols)} leaky column(s): {leaky_cols}")
print(f"Feature matrix: {X.shape[0]} patients × {X.shape[1]} features")
print(f"Feature names:  {list(X.columns)}")

**The clinical-vocabulary version of the lesson, restated:** *no feature in a predictive model is allowed to know about the future relative to the prediction time.* If your training-day-of-admission predictor uses a column that is filled in on the discharge day, you have leakage. If your sepsis-onset predictor uses a column that is set when sepsis is first treated, you have leakage. Section 3 Challenge 3 returns to the heart-failure dataset and asks the reader to *find* this column themselves — the lesson here is to demonstrate the pattern of *naming and removing* leaky columns explicitly, with rationale, so downstream readers can audit your decisions.

## 5. Class imbalance and prevalence

Heart-failure mortality in this dataset is moderately imbalanced — about a third of patients die during the observation window. *Moderately* imbalanced; not severely. The handling pattern below is the same one you would use for severely imbalanced datasets (e.g. a 5 % pneumonia prevalence on chest X-ray).

In [ ]:
n = len(y)
n_died = int((y == 1).sum())
n_alive = int((y == 0).sum())
prev = n_died / n

print(f"Total patients:    {n}")
print(f"Died (label = 1):  {n_died}  ({prev:.1%})")
print(f"Alive (label = 0): {n_alive}  ({1 - prev:.1%})")
print()
if prev < 0.10:
    print("Severe imbalance — accuracy will be misleading; require class weighting + AUC + sensitivity.")
elif prev < 0.30:
    print("Moderate imbalance — recommend stratified split and class weighting in training.")
else:
    print("Approximately balanced — no special imbalance handling needed; default training will work.")

**The class-imbalance handling decisions live downstream — in the training pipeline (notebook 02).** What this notebook is responsible for is *making the imbalance visible* and *preserving it across splits* (next section), so the training pipeline can decide how to handle it.

## 6. Stratified train / validation / test split

Three slices, three jobs (Chapter 06):

- **Training set** (60 %). The model learns from this.
- **Validation set** (20 %). Used to tune model choices — algorithm, hyperparameters, threshold.
- **Test set** (20 %). Touched once, at the very end, for the final performance estimate.

*Stratified* means: preserve the class balance (32 % mortality) across all three splits. Without stratification, a small dataset can produce splits where the test set is 20 % mortality and the training set is 35 %, and your reported test AUC underestimates real-world performance because the model trained on a different distribution than it was evaluated on.

In [ ]:
from sklearn.model_selection import train_test_split

# 60 / 20 / 20 split, stratified by outcome.
# Step 1: split off the test set (20%).
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
# Step 2: split the remaining 80% into train (60% of total) and val (20% of total).
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)  # 0.25 of 80% = 20% of total

print("Split sizes:")
print(f"  Train: {len(X_train):3d} patients ({(y_train == 1).mean():.1%} mortality)")
print(f"  Val:   {len(X_val):3d} patients ({(y_val   == 1).mean():.1%} mortality)")
print(f"  Test:  {len(X_test):3d} patients ({(y_test  == 1).mean():.1%} mortality)")
print(f"\nMortality prevalence preserved across splits → stratification worked.")

All three splits show roughly 32 % mortality. Stratification did its job. **A clinical model evaluated on a test set with the wrong prevalence is being evaluated on the wrong question.** Stratification is the cheapest defence against this; do it always.

## 7. Standardisation — making features comparable

Some models (logistic regression, SVM, neural networks) are sensitive to feature scale. Others (decision trees, random forest, gradient boosting) are not. The training pipeline (notebook 02) trains both kinds. We standardise the features here, on the *training set only*, then apply the same scaler to validation and test. **Fitting the scaler on the full dataset would itself be a form of leakage** — the test set would influence the scaling parameters.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Fit the scaler on TRAINING DATA ONLY. Then apply to validation and test.
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

# Convert back to DataFrames so feature names are preserved (matters for SHAP later).
X_train_s = pd.DataFrame(X_train_s, columns=X.columns, index=X_train.index)
X_val_s   = pd.DataFrame(X_val_s,   columns=X.columns, index=X_val.index)
X_test_s  = pd.DataFrame(X_test_s,  columns=X.columns, index=X_test.index)

print("After standardisation, training set:")
print(f"  Mean of each feature ≈ 0 (max abs deviation: {abs(X_train_s.mean()).max():.3f})")
print(f"  Std  of each feature ≈ 1 (max abs deviation: {abs(X_train_s.std() - 1).max():.3f})")

Means near 0, standard deviations near 1, across all features. The scaler has done its job on the training set. **Validation and test were transformed using the same scaler — they will not have exact mean 0 / std 1, and that is correct.** They are now in the same scale-space as the training set.

## 8. Save the prepared artefact

We save the prepared splits — *and* the scaler — to a single file the training and inference notebooks can load. Saving the scaler matters because the inference pipeline will need it to standardise *new* patients before predicting on them. **A trained model paired with a different scaler is not the same tool.** They ship together.

In [ ]:
import joblib
import os

artefact = {
    "X_train": X_train_s,
    "X_val":   X_val_s,
    "X_test":  X_test_s,
    "y_train": y_train.reset_index(drop=True),
    "y_val":   y_val.reset_index(drop=True),
    "y_test":  y_test.reset_index(drop=True),
    "scaler":  scaler,
    "feature_names": list(X.columns),
    "target_name":   TARGET,
    "removed_columns": leaky_cols,
    "data_source":   source,
}

out_path = "prepared_heart_failure.joblib"
joblib.dump(artefact, out_path)
size_kb = os.path.getsize(out_path) / 1024
print(f"Saved artefact → {out_path}  ({size_kb:.0f} KB)")
print("\nContents:")
for k, v in artefact.items():
    if hasattr(v, "shape"):
        print(f"  {k:<18} {type(v).__name__}  shape={v.shape}")
    elif isinstance(v, list):
        print(f"  {k:<18} list of {len(v)}")
    else:
        print(f"  {k:<18} {type(v).__name__}")

## 9. What is modifiable at this stage — the pipeline-decision summary

The whole point of pulling the feature pipeline out as its own notebook is that *every step above is a decision that can be revisited without retraining the model*. The list, with the clinical question each one is really asking:

| Decision | Asked clinically as | What changes if you re-decide |
| --- | --- | --- |
| Which dataset to source | Is this dataset representative of *my* patients? | The whole pipeline restarts. |
| How to handle missingness | Is *missing* itself a clinical signal here? | Different imputation strategies → different feature distributions. |
| Which columns to drop as leaky | Was this column generated *after* the prediction time? | Different AUC, sometimes by 0.10+. The headline result. |
| How to handle class imbalance | What is the prevalence in *my* deployment population? | Different threshold, different sensitivity at deployment. |
| How to split | Are there grouped patients (multiple admissions) that must stay together in one split? | Without grouped splits, leakage across train and test. |
| How to scale | Will the deployed model be linear or tree-based? | Linear models care; tree-based models don't. |

**Section 3 challenges revisit these decisions explicitly.** Challenge 3 (the same dataset) asks the reader to discover the leakage themselves; Challenge 4 (imaging) makes class imbalance the central pre-processing exercise; Challenge 5 (wearables) starts even further upstream — the data does not yet exist in tabular form.

## What this notebook produced

- `prepared_heart_failure.joblib` — train / val / test splits, the fitted `StandardScaler`, feature and target names, and metadata about what was removed and why.
- A documented decision audit trail above for every choice the pipeline made.

**Next:** notebook 02 — the training pipeline. Loads the artefact above, trains four model families, compares them on the validation set, and produces a single best model + threshold for the inference pipeline to serve.

---

*Section 2 · Notebook 1 · OCAI Booklet · Spring 2026 Edition*